# 01 — Exploratory Data Analysis

This notebook explores the ISIC 2019 dataset:
- Class distribution (bar chart + pie chart)
- Sample image grid per class
- Metadata analysis (age, sex, localisation)

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

# Add src to path so we can import project modules
sys.path.insert(0, str(Path('..') / 'src'))

from dataset import CLASS_NAMES, SkinLesionDataset

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Load ground-truth CSV

In [ ]:
GROUND_TRUTH_CSV = '../data/raw/ISIC_2019_Training_GroundTruth.csv'
METADATA_CSV     = '../data/raw/ISIC_2019_Training_Metadata.csv'
IMG_DIR          = '../data/raw/ISIC_2019_Training_Input'

gt_df = pd.read_csv(GROUND_TRUTH_CSV)
print(f'Ground-truth shape: {gt_df.shape}')
gt_df.head()

## 2. Class distribution

In [ ]:
# One-hot → integer label
label_cols = [c for c in CLASS_NAMES if c in gt_df.columns]
gt_df['label'] = gt_df[label_cols].values.argmax(axis=1)
gt_df['class_name'] = gt_df['label'].map({i: n for i, n in enumerate(CLASS_NAMES)})

counts = gt_df['class_name'].value_counts().reindex(CLASS_NAMES)
print(counts)
print(f'\nTotal images: {counts.sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].bar(counts.index, counts.values, color=sns.color_palette('muted', len(CLASS_NAMES)))
axes[0].set_title('Class Distribution — Bar Chart')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontsize=9)

# Pie chart
axes[1].pie(
    counts.values,
    labels=counts.index,
    autopct='%1.1f%%',
    colors=sns.color_palette('muted', len(CLASS_NAMES)),
    startangle=140,
)
axes[1].set_title('Class Distribution — Pie Chart')

plt.tight_layout()
plt.savefig('../outputs/results/class_distribution.png', dpi=150)
plt.show()

## 3. Sample images per class

In [ ]:
N_SAMPLES = 4  # images per class
fig, axes = plt.subplots(len(CLASS_NAMES), N_SAMPLES, figsize=(N_SAMPLES * 3, len(CLASS_NAMES) * 3))

for row_idx, cls in enumerate(CLASS_NAMES):
    class_imgs = gt_df[gt_df['class_name'] == cls]['image_id'].tolist()
    samples = class_imgs[:N_SAMPLES]
    for col_idx, img_id in enumerate(samples):
        ax = axes[row_idx][col_idx]
        img_path = Path(IMG_DIR) / f'{img_id}.jpg'
        if img_path.exists():
            ax.imshow(Image.open(img_path))
        else:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center')
        ax.axis('off')
        if col_idx == 0:
            ax.set_ylabel(cls, rotation=0, labelpad=50, fontsize=11, fontweight='bold')

plt.suptitle('Sample Images per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/results/sample_images.png', dpi=150)
plt.show()

## 4. Metadata analysis

In [ ]:
meta_df = pd.read_csv(METADATA_CSV)
print(f'Metadata shape: {meta_df.shape}')
meta_df.head()

In [ ]:
# Merge labels with metadata
merged = pd.merge(gt_df[['image_id', 'class_name']], meta_df, on='image_id', how='left')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age distribution
if 'age_approx' in merged.columns:
    merged['age_approx'].dropna().hist(bins=20, ax=axes[0], edgecolor='black')
    axes[0].set_title('Age Distribution')
    axes[0].set_xlabel('Age')

# Sex distribution
if 'sex' in merged.columns:
    merged['sex'].value_counts().plot.bar(ax=axes[1], color=['steelblue', 'salmon'])
    axes[1].set_title('Sex Distribution')
    axes[1].tick_params(axis='x', rotation=0)

# Localisation
if 'anatom_site_general' in merged.columns:
    merged['anatom_site_general'].value_counts().plot.barh(ax=axes[2])
    axes[2].set_title('Anatomical Localisation')

plt.tight_layout()
plt.savefig('../outputs/results/metadata_analysis.png', dpi=150)
plt.show()